In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import inventory as I
import manifest as M
import assemble as A
import concepts as C

smh_dir = Path("/home/jormond/Documents/DS217/users/jormond/SMH")
rbh_dir = Path("/home/jormond/Documents/DS217/users/jormond/RBH")
gosh_dir = Path("/home/jormond/Documents/DS217/users/jormond/GOSH")

### What is in the assembled tables
Prints, for each site, every assembled table with its row count and column names (no values). RBH tables are mostly wide (one column per measurement) and GOSH has one file per variable, so these names are what the concept map will be written from.

In [ ]:
assembled_dir = open_picu_dir / "derived" / "assembled"
for site in ["RBH", "GOSH"]:
    C.list_sources(assembled_dir, site)

### Cross-site concept map
Lists the long GOSH label columns first (so GOSH blood gases, fluids and drugs can be mapped), then measures every source in the map at all three sites: stays with a value, rows, share numeric and the 5th, median and 95th percentile. The percentiles show unit differences between sites. Counts and ranges only. This reads every assembled table, so it takes a few minutes.

In [ ]:
C.list_values(assembled_dir, "GOSH", "merged_vent_data__vent_Ventilation_mode_flowsheet_rows",
              "res_value", where=("var_type", "Ventilation"))

In [ ]:
G = "GOSH"
C.list_values(assembled_dir, G, "merged_gas_data__merged_blood_gasses", "standard_variable")
C.list_values(assembled_dir, G, "merged_balance__Net_Bal", "shortLabel")
C.list_values(assembled_dir, G, "merged_balance__Total_fluids", "shortLabel")
C.list_values(assembled_dir, G, "merged_vent_data__vent_Ventilation_mode_flowsheet_rows", "DisplayName")

In [ ]:
pooled_static = pd.read_parquet(open_picu_dir / "derived" / "static" / "pooled_static.parquet")
detail, summary = C.build(assembled_dir, static=pooled_static,
                          save_to=open_picu_dir / "derived" / "concepts" / "concept_map.csv")
pd.set_option("display.max_rows", 100)
print(summary.drop(columns="note").to_string(index=False))

### Coverage by admission year
For a few concepts, the percentage of each site's stays with the concept in each admission year. A source that only exists in some years (for example a system change) shows as a step.

In [ ]:
chk = ["bg_ph", "vent_mode", "tidal_volume", "ie_ratio", "net_balance", "etco2", "abp_sys", "fio2"]
yr, piv = C.by_year(assembled_dir, pooled_static, chk)
print(piv.to_string())

In [ ]:
A.list_tables("GOSH", gosh_dir, include=r"raw_lab_values|lab_components|Archived", exclude="")

In [ ]:
import pyarrow.parquet as pq
pq.ParquetFile(gosh_dir / "gosh_mech_vent_31March2026" / "final_gosh_dataset.parquet").schema.names

In [ ]:
yr, piv = C.by_year(assembled_dir, pooled_static, list(C.CONCEPTS), sites=("GOSH",))

def share(a, b):
    d = yr[yr.year.between(a, b)].groupby("concept")[["stays", "n"]].sum()
    return (100 * d.stays / d.n).round(1)

era = pd.DataFrame({"pre_epic_pct": share(2013, 2018), "post_epic_pct": share(2020, 2022)})
print(era[(era.pre_epic_pct >= 50) & (era.post_epic_pct < 0.5 * era.pre_epic_pct)].to_string())

In [ ]:
cols = ["hour_timestamp", "pH", "ventMode", "TV_measured", "IERatio"]
old = pd.read_parquet(gosh_dir / "gosh_mech_vent_31March2026" / "final_gosh_dataset.parquet", columns=cols)
old["year"] = pd.to_datetime(old["hour_timestamp"], errors="coerce").dt.year
print(old.groupby("year")[["pH", "ventMode", "TV_measured", "IERatio"]].agg(lambda s: round(100 * s.notna().mean(), 1)).to_string())

### Does the old GOSH parquet hold post-2019 data we are missing?
`final_gosh_dataset.parquet` is an older hourly dataset built from the GOSH sources. For every column, this shows how many hourly rows have a value before and after the CareVue to Epic switch (about 20 April 2019). If blood gas, vent mode, tidal volume or I:E ratio have values after 2019 here, they exist in a source we have not assembled. Counts only.

In [ ]:
old_path = gosh_dir / "gosh_mech_vent_31March2026" / "final_gosh_dataset.parquet"
era = C.era_columns(old_path)
print(era.sort_values("post_pct").to_string())

In [ ]:
g = pooled_static[pooled_static.site == "GOSH"]
print((pd.to_datetime(g["t0"]) >= "2019-04-20").sum(), len(g))

### Which features does each stay have?
Builds a stay-by-feature table (has a value, or not) so users can choose a feature set and the stays that have it. The pattern of available features could itself identify a stay, so this also counts how many stays share their pattern with fewer than 10 others, per feature and per group. Row-level: saved in derived/ only.

In [ ]:
avail = C.availability(assembled_dir, pooled_static,
                       save_to=open_picu_dir / "derived" / "concepts" / "stay_availability.parquet")
print("per feature:", C.pattern_uniqueness(avail))
print("per group:  ", C.pattern_uniqueness(avail, by_group=True))

In [ ]:
summary = pd.read_csv(open_picu_dir / "derived" / "concepts" / "concept_summary.csv")
avail = pd.read_parquet(open_picu_dir / "derived" / "concepts" / "stay_availability.parquet")

In [ ]:
for pct in (90, 70, 50):
    cols = C.survivors(summary, pct)["concept"].tolist()
    print(pct, len(cols), C.pattern_uniqueness(avail[cols]))
cols = C.survivors(summary, 50, exclude_era=True)["concept"].tolist()
print("50, era-safe", len(cols), C.pattern_uniqueness(avail[cols]))

In [ ]:
cols = C.survivors(summary, 50)["concept"].tolist()
print(C.rare_share(avail[cols], pooled_static, by=["site"]))
print(C.rare_share(avail[cols], pooled_static, by=["site", "year"]).unstack("site"))

## Drugs: names in the data

The concept map has no drugs yet. The SMH drug IDs in `smh_mappings.py` were taken from the extraction SQL, and four infusion IDs were added by reading `Drug_terse`. Before mapping drugs across sites, list the drug names in the SMH medication, infusion and antibiotic tables, with the number of stays each appears in, and check them against the ID maps. Counts of stays only; no rows leave.

In [ ]:
assembled_dir = open_picu_dir / "derived" / "assembled"
C.list_sources(assembled_dir, "SMH", show_columns=True)

Number of stays per drug name, for each SMH drug table. Names with fewer than 10 stays are not listed.

In [ ]:
for t in ["medications", "infusions"]:
    print(t)
    print(C.list_values(assembled_dir, "SMH", t, "Drug_terse", min_stays=10, top=80))

In [ ]:
C.list_values(assembled_dir, "SMH", "infusions", "Drug_terse", min_stays=1, top=80)

In [ ]:
import pandas as pd
m = pd.read_parquet(assembled_dir / "SMH" / "ai_picu_medications_v2.parquet", columns=["Drug_terse", "utcChartTime"])
f = m[m["Drug_terse"] == "Furosemide"]["utcChartTime"].astype(str)
print(f.str.len().value_counts().head())
t = pd.to_datetime(f, errors="coerce", format="mixed")
print(t.isna().sum(), t.dt.month.max(), t.dt.day.max())

## Drugs: RBH and GOSH

Same check as for SMH. First find the drug tables and the column that holds the drug name at RBH and GOSH, then list the names with the number of stays each appears in. The aim is a cross-site drug table with merged spellings (for example the two morphine spellings at SMH). RBH has `infusion`, `meds`, `sedation` and `sedation_old` (the last two overlap in time); GOSH has `filtered_ALL_drugs` and `intrv_details` (pre-Epic) and `medsG_medication_admins` (Epic only), so the drug lists are expected to differ by era.

In [ ]:
for S in ["RBH", "GOSH"]:
    C.list_sources(assembled_dir, S, show_columns=True)

Number of stays per drug name for each RBH drug table. Names seen in fewer than 10 stays are not listed.

In [ ]:
for t in ["ai_picu_infusion", "ai_picu_meds", "ai_picu_sedation", "ai_picu_sedation_old"]:
    print(t)
    C.list_values(assembled_dir, "RBH", t, "Drug", min_stays=10, top=80)

Same for GOSH. The drug files are `filtered_ALL_drugs_merged`, `filtered_CI_drugs_merged` (8.1M rows, probably continuous infusions) and `intrv_details`, which name the drug in `shortLabel`; `medsG_medication_admins` (Epic only) and the antibiotics table use `drug_name`. The pre-Epic and Epic tables are listed separately, because names and coverage differ by era.

In [ ]:
_D = "merged_drugs__"
for t, col in [(_D + "filtered_ALL_drugs_merged", "shortLabel"),
               (_D + "filtered_CI_drugs_merged", "shortLabel"),
               (_D + "intrv_details", "shortLabel"),
               (_D + "medsG_medication_admins", "drug_name"),
               ("July2026extraction_by_alex__pAIC__merged_antibiotics__antibiotics", "drug_name")]:
    print(t)
    C.list_values(assembled_dir, "GOSH", t, col, min_stays=10, top=80)

Where does each GOSH drug source start and stop? Number of stays per year for the main drugs in the pre-Epic infusion table (`filtered_CI_drugs_merged`) and the Epic-only administration table (`medsG_medication_admins`). Years with fewer than 10 stays are blanked. If the infusion table stops at the Epic switch (April 2019), GOSH has no infusion rates after it.

In [ ]:
_D = "merged_drugs__"
C.names_by_year(assembled_dir, "GOSH", _D + "filtered_CI_drugs_merged", "shortLabel", "observation_datetime", top=8)

In [ ]:
C.names_by_year(assembled_dir, "GOSH", _D + "medsG_medication_admins", "drug_name", "observation_datetime", top=8)

In [ ]:
C.names_by_year(assembled_dir, "GOSH", _D + "filtered_ALL_drugs_merged", "shortLabel", "observation_datetime", top=8)

## Drugs: coverage and the GOSH Epic era

Build the drug concepts (any record of the drug in the stay) and measure them like the other features. Then check by admission year whether GOSH coverage drops at the Epic switch (April 2019). Before it the infusion tables hold rates; after it only `medsG` administration records exist, with no rates. A fall in the sedatives and analgesics would mean the GOSH Epic-era infusion data is missing, as for blood gas.

In [ ]:
drugs = [n for n in C.CONCEPTS if n.startswith("drug_")]
detail_d, summary_d = C.build(assembled_dir, static=pooled_static, concepts=drugs)
print(summary_d[["concept", "RBH_pct", "SMH_pct", "GOSH_pct"]].to_string())

Percent of GOSH stays with each drug, by admission year. A step down from 2019 means the Epic-era records do not capture the drug.

In [ ]:
out, piv = C.by_year(assembled_dir, pooled_static, drugs, sites=["GOSH"])
print(piv.round(0).to_string())

Check whether the RBH `IE_Ratio` and `HFOV_Frequency` columns hold real values or a default fill. Both show 100% of stays. List the most common values with the number of stays each appears in, and compare the ratio columns (`Insp_Ratio`, `Exp_Ratio`).

In [ ]:
for c in ["IE_Ratio", "HFOV_Frequency", "Insp_Ratio", "Exp_Ratio"]:
    C.list_values(assembled_dir, "RBH", "ai_picu_vent", c, min_stays=10, top=8)

In [ ]:
print(sorted(p.name for p in (open_picu_dir / "notebooks").glob("*.ipynb")))

# common feature sets

In [ ]:
for m in [90, 70, 50]:
    print(m, len(C.survivors(summary, m)), len(C.survivors(summary, m, exclude_era=True)))

## availability and patterns

In [ ]:
derived_dir = open_picu_dir / "derived"
avail = C.availability(assembled_dir, pooled_static, save_to=derived_dir / "concepts" / "stay_availability.parquet")
print(avail.shape)
for m in [90, 70, 50]:
    cols = C.survivors(summary, m)["concept"].tolist()
    print(m, len(cols), C.pattern_uniqueness(avail[cols]))
cols = C.survivors(summary, 50, exclude_era=True)["concept"].tolist()
print("50 era-safe", len(cols), C.pattern_uniqueness(avail[cols]))
print("by group", C.pattern_uniqueness(avail, by_group=True))

Rerun after fixing empty columns and the `.:.` placeholder in `concepts.py`. Re-measure all concepts, then recount the common feature sets and the availability patterns.

In [ ]:
detail, summary = C.build(assembled_dir, static=pooled_static,
                          save_to=open_picu_dir / "derived" / "concepts" / "concept_map.csv")
print(summary.loc[summary["concept"].isin(["ie_ratio", "hfov_frequency"]),
                  ["concept", "RBH_stays", "RBH_pct", "SMH_pct", "GOSH_pct"]].to_string())

In [ ]:
for m in [90, 70, 50]:
    print(m, len(C.survivors(summary, m)), len(C.survivors(summary, m, exclude_era=True)))

In [ ]:
avail = C.availability(assembled_dir, pooled_static, save_to=derived_dir / "concepts" / "stay_availability.parquet")
print(avail.shape)
for m in [90, 70, 50]:
    cols = C.survivors(summary, m)["concept"].tolist()
    print(m, len(cols), C.pattern_uniqueness(avail[cols]))
cols = C.survivors(summary, 50, exclude_era=True)["concept"].tolist()
print("50 era-safe", len(cols), C.pattern_uniqueness(avail[cols]))
print("by group", C.pattern_uniqueness(avail, by_group=True))

In [ ]:
cols = C.survivors(summary, 50)["concept"].tolist()
print(C.rare_share(avail[cols], pooled_static, by=["site"]))
print(C.rare_share(avail[cols], pooled_static, by=["site", "year"]).unstack("site"))

Keep a dated copy of the concept summary and detail, so later reruns can be compared with these reference numbers. Counts and ranges only; the files stay in `derived/`.

In [ ]:
out_dir = open_picu_dir / "derived" / "concepts"
summary.to_csv(out_dir / "concept_summary_2026-11-18.csv", index=False)
detail.to_csv(out_dir / "concept_detail_2026-11-18.csv", index=False)
print(summary.loc[summary["concept"].isin(["intake_total", "output_total"]),
                  ["concept", "RBH_stays", "RBH_pct", "SMH_pct", "GOSH_pct"]].to_string())

## Units and value ranges

Compare the 5th percentile, median and 95th percentile of each numeric concept across the three sites. Concepts whose medians differ by more than three times, or change sign, are flagged as a likely unit difference (kPa/mmHg, g/dL/g/L, fraction/percent). The flags are a list to check, not a decision; conversions are applied at harmonisation.

In [ ]:
pd.set_option("display.width", 200)
rng = C.compare_ranges(detail)
print(rng[rng["unit_flag"]].to_string(index=False))

In [ ]:
print(rng[~rng["unit_flag"]].to_string(index=False))

## Fluids: hourly, 24-hour or cumulative?

The three GOSH balance files (`Net_Bal`, `Net_Bal_24`, `Total_fluids`) carry a start and end time per row. List each label with the length of its intervals, to tell hourly values from 24-hour totals and from totals cumulated over the stay. This decides the common fluid definition across sites.

In [ ]:
for t in ["merged_balance__Net_Bal", "merged_balance__Net_Bal_24", "merged_balance__Total_fluids"]:
    C.interval_check(assembled_dir, "GOSH", t, "shortLabel", "start_datetime", "end_datetime")

In [ ]:
ids = [i for v in C.SM.IO.values() for i in v]
C.SM.describe_ids(assembled_dir, "io", ids, "hourTotal")
C.SM.describe_ids(assembled_dir, "io", ids, "cumTotal")

### gosh caboodle input

In [ ]:
print(detail.loc[detail["source"].str.contains("caboodle", na=False),
                 ["concept", "source", "stays", "rows", "p5", "median", "p95"]].to_string(index=False))

Two checks on the SMH `io` table. First, is `hourTotal` the difference between consecutive `cumTotal` values (so hourly amounts can be derived from cumulative-only sources such as GOSH)? Second, which components add up to total intake and total output, hour by hour?

In [ ]:
C.io_diff_check(assembled_dir)

In [ ]:
C.io_reconcile(assembled_dir, ["2045", "113860"], ["1851", "1912", "2052", "2047", "2101"])
C.io_reconcile(assembled_dir, ["2073", "113859", "113961"], ["1945", "1949"])

Two checks before defining the GOSH hourly fluid features. First, how much of the caboodle hourly data is zero, and do the hourly values add up to the daily columns? Second, are the pre-Epic GOSH 24-hour totals hourly rows that reset daily, as at SMH? If so, hourly amounts can be derived by differencing.

In [ ]:
C.caboodle_check(assembled_dir)

In [ ]:
_B = "merged_balance__"
C.cum_check(assembled_dir, "GOSH", _B + "Total_fluids", "shortLabel", ["Total Fluids In (24hrs)", "Total Fluids Out (24hrs)"], "chartTime", "cumTotal")
C.cum_check(assembled_dir, "GOSH", _B + "Net_Bal", "shortLabel", ["LOS Net Body Balance"], "chartTime", "cumTotal")

In [ ]:
C.caboodle_check(assembled_dir)
C.io_diff_check(assembled_dir)

## Value checks before cleaning

For the concepts flagged in the range comparison, count the share of values that are non-numeric, exactly zero or negative, per site and source. This finds placeholder zeros (RBH PIP, mean airway pressure and set rate; GOSH monitor respiratory rate), text where a number is expected (GOSH CRT and tidal volume, ratios), and negative values that may have been lost (GOSH base excess). Counts only.

In [ ]:
chk = C.value_checks(assembled_dir, ["bg_base_excess", "bg_hb", "bg_hct", "haemoglobin", "nibp_dia", "abp_dia", "pip",
                                     "mean_airway_pressure", "rr_set", "resp_rate_monitor", "tidal_volume", "crt",
                                     "ie_ratio", "airway_device", "insp_time", "bilirubin"])
print(chk.to_string(index=False))

The units GOSH states for the variables whose ranges looked different across sites (bilirubin, haemoglobin, capillary refill, tidal volume, I:E ratio).

In [ ]:
for t in [C._L + "TotalBilirubin_merged", C._L + "Haemoglobin_merged", C._L + "merged_Hb", C._V + "Capillary_merged",
          C._VENT + "merged_tidal_volumes", C._VENT + "merged_IE_ratios"]:
    C.list_values(assembled_dir, "GOSH", t, "unit_of_measure", min_stays=10, top=8)

## Checks on the specific problems

Base excess against pH from the same sample (does GOSH lack the acidotic samples, or have their sign stripped?); the shape of the text values where a number is expected; value ranges by unit label for haemoglobin and haematocrit; and whether blood pressure readings are in a possible order (diastolic below mean below systolic).

In [ ]:
C.be_ph_check(assembled_dir, "RBH")
C.be_ph_check(assembled_dir, "GOSH")

In [ ]:
for c in ["haemoglobin", "bilirubin", "tidal_volume", "crt", "ie_ratio"]:
    for s in ["RBH", "SMH", "GOSH"]:
        C.value_shapes(assembled_dir, c, s)

In [ ]:
C.percentiles_by(assembled_dir, "GOSH", "merged_lab_values__Haemoglobin_merged", "res_value", "unit_of_measure")
C.percentiles_by(assembled_dir, "GOSH", "merged_gas_data__merged_blood_gasses", "res_value", "standard_variable", min_rows=20000)

In [ ]:
for kind in ["NIBP", "ABP"]:
    C.bp_order_check(assembled_dir, "RBH", kind)
    C.bp_order_check(assembled_dir, "GOSH", kind)

### I:E ratio parts

The SMH I:E ratio comes from two IDs (4185 and 4186) and the values are numeric with a median of 1, which looks like inspiratory and expiratory parts mixed together (RBH has `Insp_Ratio` of 1.0 and `Exp_Ratio` of 2.0 as separate columns). Describe each SMH ID separately, and list the GOSH `source` labels to see whether GOSH separates the two parts.

In [ ]:
C.SM.describe_ids(assembled_dir, "vent", ["4185", "4186"], "terseForm")
C.list_values(assembled_dir, "GOSH", C._VENT + "merged_IE_ratios", "source", min_stays=10)

### I:E ratio values

The most common values (with the number of stays) for the two SMH IDs, and for each GOSH source label, to see what the numbers mean before defining one ratio across sites.

In [ ]:
for i in ["4185", "4186"]:
    C.list_values(assembled_dir, "SMH", "vent", "terseForm", min_stays=50, top=12, where=("interventionId", i))
for src in ["I:E Ratio.Inspiratory", "I:E Ratio.Expiratory", "I:E Ratio Setting"]:
    C.list_values(assembled_dir, "GOSH", C._VENT + "merged_IE_ratios", "res_value", min_stays=50, top=10, where=("source", src))

### I:E ratio parts: pairing and settings

Check whether the SMH I:E rows come in pairs at the same time (one row is the value 1, the other the expiratory part), and where the GOSH "I:E Ratio Setting" values are stored, since its `res_value` is empty.

In [ ]:
C.ie_pair_check(assembled_dir)
C.list_values(assembled_dir, "GOSH", C._VENT + "merged_IE_ratios", "comment", min_stays=50, top=10, where=("source", "I:E Ratio Setting"))

### GOSH diastolic blood pressure, again

Rerun the blood pressure order check, which now also reports how often diastolic is exactly equal to systolic or to the mean. If diastolic equals systolic in many readings, the diastolic file holds copies of systolic.

In [ ]:
for kind in ["NIBP", "ABP"]:
    C.bp_order_check(assembled_dir, "GOSH", kind)

### SMH laboratory values: numeric stays

About half the SMH haemoglobin and bilirubin rows hold the word "Blood", not a value. Rebuild the concept detail and compare the stays that have any value (`stays`) with those that have a numeric value (`stays_numeric`), to check that coverage is not inflated by stays with only "Blood" rows.

In [ ]:
detail, summary = C.build(assembled_dir, static=pooled_static)
lab = detail[(detail["site"] == "SMH") & (detail["status"] == "ok") & (detail["stays"] != detail["stays_numeric"])]
print(lab[["concept", "source", "stays", "stays_numeric", "numeric_pct"]].to_string(index=False))